In [9]:
import geopandas as gpd
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
BOUNDARIES_DIR = DATA_DIR / "boundaries" / "processed"

TOC_PATH = BOUNDARIES_DIR / "toc_clean.gpkg"          # your TOC layer
VILLAGE_PATH = BOUNDARIES_DIR / "village_clean.gpkg"  # Phoenix village layer

toc = gpd.read_file(TOC_PATH)
villages = gpd.read_file(VILLAGE_PATH)

# Make sure CRS matches (EPSG:2868)
villages = villages.to_crs(toc.crs)

print(toc.crs, villages.crs)

EPSG:2868 EPSG:2868


In [10]:
toc = toc.reset_index().rename(columns={"index": "index_right"})
toc = toc.set_index("index_right")

toc.index.name

'index_right'

In [11]:
vill_sub = villages[["NAME", "geometry"]]

toc_sub = toc[["APPLICABIL", "geometry"]]
toc_sub = toc_sub.reset_index()

toc_vill_intersections = gpd.overlay(
    toc_sub,   # index_right + geometry
    vill_sub,  # NAME + geometry
    how="intersection"
)

# Area in acres (assuming CRS is in feet → area in sq ft)
toc_vill_intersections["intersect_acres"] = (
    toc_vill_intersections.geometry.area / 43560.0
)

toc_vill_intersections.head()

,index_right,APPLICABIL,NAME,geometry,intersect_acres
0,0,TOD District - Gateway,Central City,"POLYGON ((663434.469 894514.048, 663444.682 89...",1968.548833
1,0,TOD District - Gateway,Camelback East,"POLYGON ((672618.313 894544.27, 672760.169 894...",450.395529
2,1,TOD District - Eastlake Garfield,Central City,"POLYGON ((654764.736 895069.8, 654771.953 8954...",1220.462979
3,2,TOD District - Midtown,Encanto,"POLYGON ((654767.631 907434.551, 654767.356 90...",1283.000217
4,3,TOD District - Uptown,Camelback East,"POLYGON ((654791.762 915480.496, 654790.624 91...",0.000105


In [12]:
toc_vill_largest = (
    toc_vill_intersections
    .sort_values("intersect_acres", ascending=False)
    .drop_duplicates(subset="index_right", keep="first")
    .loc[:, ["index_right", "NAME", "intersect_acres"]]
)

In [14]:
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

toc_stats = pd.read_csv(OUTPUTS_DIR / "toc_land_value_stats.csv")
toc_stats.head(8)

,index_right,APPLICABIL,total_taxable_value,total_full_cash,total_land_value,total_acres,parcel_count,taxable_value_per_acre
0,0.0,TOD District - Gateway,285984758.0,3.718919e+09,8.830583e+08,2346.119575,3259,121896.923347
1,1.0,TOD District - Eastlake Garfield,207168885.0,3.014334e+09,8.307006e+08,1015.214786,3323,204064.093422
2,2.0,TOD District - Midtown,468187974.0,7.310771e+09,1.939664e+09,1053.891243,4552,444246.953429
3,3.0,TOD District - Uptown,171863623.0,3.589014e+09,9.414611e+08,1040.744605,3746,165135.252405
4,4.0,TOD District - Solano,91072483.0,1.978354e+09,4.115623e+08,848.835227,3414,107291.120997
5,5.0,TOD District - 19North,168607085.0,3.933583e+09,8.894743e+08,1592.456991,5306,105878.580075
6,6.0,50th Street Station Area,74683387.0,1.238704e+09,2.995826e+08,754.371294,270,99000.833669
7,7.0,Capitol Extension Area,148353832.0,1.961782e+09,5.294767e+08,807.263067,2396,183773.837815


In [16]:
toc_stats_village = toc_stats.merge(
    toc_vill_largest,
    on="index_right",
    how="left"
)

toc_stats_village.head(8)

,index_right,APPLICABIL,total_taxable_value,total_full_cash,total_land_value,total_acres,parcel_count,taxable_value_per_acre,NAME,intersect_acres
0,0.0,TOD District - Gateway,285984758.0,3.718919e+09,8.830583e+08,2346.119575,3259,121896.923347,Central City,1968.548833
1,1.0,TOD District - Eastlake Garfield,207168885.0,3.014334e+09,8.307006e+08,1015.214786,3323,204064.093422,Central City,1220.462979
2,2.0,TOD District - Midtown,468187974.0,7.310771e+09,1.939664e+09,1053.891243,4552,444246.953429,Encanto,1283.000217
3,3.0,TOD District - Uptown,171863623.0,3.589014e+09,9.414611e+08,1040.744605,3746,165135.252405,Alhambra,918.483468
4,4.0,TOD District - Solano,91072483.0,1.978354e+09,4.115623e+08,848.835227,3414,107291.120997,Alhambra,1100.265433
5,5.0,TOD District - 19North,168607085.0,3.933583e+09,8.894743e+08,1592.456991,5306,105878.580075,Alhambra,1127.773957
6,6.0,50th Street Station Area,74683387.0,1.238704e+09,2.995826e+08,754.371294,270,99000.833669,Camelback East,595.809380
7,7.0,Capitol Extension Area,148353832.0,1.961782e+09,5.294767e+08,807.263067,2396,183773.837815,Central City,1112.954359


In [17]:
out_csv = OUTPUTS_DIR / "toc_land_value_stats_with_village.csv"
toc_stats_village.to_csv(out_csv, index=False)

print("Saved:", out_csv)

Saved: c:\Users\arjav\DevSource\toc-performance-dashboard\outputs\toc_land_value_stats_with_village.csv
